In [8]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

In [9]:
df = pd.read_csv('../../Datos/df_general.csv')

In [10]:
isolated_spoof_id = 'A14'

print(f"--- EXPERIMENTO TOTAL: Train (2k balanceado) vs Test (~88k restantes) ---")

# 2. Separar las muestras por tipo 
df_reales = df[df['Key'] != 'spoof']
df_falsos = df[df['Key'] == 'spoof']

# 3. Muestras del ataque DESCONOCIDO (Van TODAS al test)
falsos_desconocidos_test = df_falsos[df_falsos['Spoofing_ID'] == isolated_spoof_id]

# 4. Muestras de los ataques CONOCIDOS
falsos_conocidos_pool = df_falsos[df_falsos['Spoofing_ID'] != isolated_spoof_id]

# --- 5. REPARTO EXACTO SIN PERDER NI UNA MUESTRA ---

# A) Voces Reales: 1000 para Train, el resto para Test
reales_train, reales_test = train_test_split(
    df_reales, 
    train_size=1000, 
    random_state=42
)

# B) Ataques Conocidos: 1000 para Train (Estratificados), el resto enorme para Test
falsos_conocidos_train, falsos_conocidos_test = train_test_split(
    falsos_conocidos_pool, 
    train_size=1000, 
    stratify=falsos_conocidos_pool['Spoofing_ID'], 
    random_state=42
)

--- EXPERIMENTO TOTAL: Train (2k balanceado) vs Test (~88k restantes) ---


In [11]:
# 6. Construir los DataFrames Finales
# Train: 1000 reales + 1000 ataques conocidos
df_train = pd.concat([reales_train, falsos_conocidos_train]).sample(frac=1, random_state=42).reset_index(drop=True)

# Test: Todo lo demás (Reales restantes + Conocidos restantes + Todos los de A14)
df_test = pd.concat([reales_test, falsos_conocidos_test, falsos_desconocidos_test]).sample(frac=1, random_state=42).reset_index(drop=True)

# 7. Separar X e y
columnas_a_excluir = ['file_name', 'User_ID', 'Spoofing_ID', 'Key']

X_train = df_train.drop(columns=columnas_a_excluir)
y_train = df_train['Key']

X_test = df_test.drop(columns=columnas_a_excluir)
y_test = df_test['Key']

print(f"\n[INFO] Muestras en Entrenamiento: {len(y_train)}")
print(f"[INFO] Muestras en Test: {len(y_test)}")
print("-" * 50)

# 8. Entrenar Modelo
rf_model = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 9. Evaluar Global
y_pred = rf_model.predict(X_test)
df_test['Prediccion'] = y_pred  # Guardamos la predicción para el análisis detallado

print("\n=== RESULTADOS GLOBALES EN EL TEST SET COMPLETO ===")
print(classification_report(y_test, y_pred))


[INFO] Muestras en Entrenamiento: 2000
[INFO] Muestras en Test: 92632
--------------------------------------------------

=== RESULTADOS GLOBALES EN EL TEST SET COMPLETO ===
              precision    recall  f1-score   support

    bonafide       0.28      0.87      0.42      6950
       spoof       0.99      0.82      0.90     85682

    accuracy                           0.82     92632
   macro avg       0.63      0.84      0.66     92632
weighted avg       0.93      0.82      0.86     92632



In [ ]:
# --- 10. ANÁLISIS DETALLADO (La clave de este experimento) ---
print("\n=== RECALL A14 ===")

# Precisión A14 (Ataque DESCONOCIDO)
mask_desconocido = df_test['Spoofing_ID'] == isolated_spoof_id
aciertos_desconocido = sum(df_test[mask_desconocido]['Key'] == df_test[mask_desconocido]['Prediccion'])
total_desconocido = sum(mask_desconocido)
print(f"-> Ataque (A14) bloqueado: {aciertos_desconocido}/{total_desconocido} ({(aciertos_desconocido/total_desconocido)*100:.2f}%)")


=== DESGLOSE DE EFECTIVIDAD (RECALL) POR TIPO DE AUDIO ===
-> Ataque (A14) bloqueado: 4818/4914 (98.05%)
